In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# Define the URLs for the seasons you want to scrape
season_urls = {
    #"2013-2014": "https://www.flashscore.es/futbol/inglaterra/premier-league-2013-2014/resultados/",
    #"2014-2015": "https://www.flashscore.es/futbol/inglaterra/premier-league-2014-2015/resultados/",
    #"2015-2016": "https://www.flashscore.es/futbol/inglaterra/premier-league-2015-2016/resultados/",
    #"2016-2017": "https://www.flashscore.es/futbol/inglaterra/premier-league-2016-2017/resultados/",
    "2017-2018": "https://www.flashscore.es/futbol/inglaterra/premier-league-2017-2018/resultados/",
    "2018-2019": "https://www.flashscore.es/futbol/inglaterra/premier-league-2018-2019/resultados/",
    "2019-2020": "https://www.flashscore.es/futbol/inglaterra/premier-league-2019-2020/resultados/",
    "2020-2021": "https://www.flashscore.es/futbol/inglaterra/premier-league-2020-2021/resultados/",
    "2021-2022": "https://www.flashscore.es/futbol/inglaterra/premier-league-2021-2022/resultados/",
    "2022-2023": "https://www.flashscore.es/futbol/inglaterra/premier-league-2022-2023/resultados/",
    "2023-2024": "https://www.flashscore.es/futbol/inglaterra/premier-league-2023-2024/resultados/"
}

BATCH_SIZE = 380  # Restart browser after this many matches

# --- Global lists to store all collected data ---
all_commentary_data = []

# --- Main Scraping Loop by Season ---
for season, base_url in season_urls.items():
    print(f"\n--- Starting to scrape season: {season} ---")
    # Use Firefox WebDriver
    driver = webdriver.Firefox()  # Start a new Firefox browser for each season
    wait = WebDriverWait(driver, 3)  # Initialize WebDriverWait for the new driver

    driver.get(base_url)

    # Step 1: Click "show more" repeatedly to load all matches
    while True:
        try:
            # Look for the specific "show more" button for the season results table
            show_more = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)  # Wait for content to load after clicking
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season}. All matches loaded.")
            break
        except Exception as e:
            print(f"An unexpected error occurred while clicking 'show more' for {season}: {e}")
            break  # Exit loop if unexpected error

    # Step 2: Extract all match links and details from the season results page
    current_season_match_details = []
    try:
        # Wait for at least one match row to be present
        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]"))
        )
        match_elements_rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]")

        for match_row_element in match_elements_rows:
            match_link = ""
            home_team = "N/A"
            away_team = "N/A"
            match_date_time = "N/A"  # This will capture "DD.MM. HH:MM"

            try:
                # The match link is typically found in an 'a' tag with class 'eventRowLink'
                link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
                href = link_element.get_attribute("href")
                if href and "/#/resumen-del-partido" in href:
                    match_link = href
            except NoSuchElementException:
                pass  # Link not found for this row, skip

            if not match_link:
                continue  # Skip if no valid match link is found

            try:
                # Home team name - adapt XPath if needed based on the current Flashscore DOM
                home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]")
                home_team = home_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                # Away team name - adapt XPath if needed based on the current Flashscore DOM
                away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]")
                away_team = away_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                # Date and Time - adapt XPath if needed
                date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
                match_date_time = date_element.text.strip()
            except NoSuchElementException:
                pass

            current_season_match_details.append({
                "Season": season,
                "Match URL": match_link,
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time  # Use this for consistency
            })
    except TimeoutException:
        print(f"Timeout while waiting for match rows for {season}. No matches found or loaded slowly.")
    except Exception as e:
        print(f"An unexpected error occurred while extracting match links for {season}: {e}")

    print(f"Found {len(current_season_match_details)} match links with details for {season}.")
    driver.quit()  # Close the browser after collecting all match links for the season

    # Step 3: Scrape commentaries and statistics in batches
    for i in range(0, len(current_season_match_details), BATCH_SIZE):
        print(f"\n[INFO] Restarting browser... Batch starting from match {i+1} of {len(current_season_match_details)}")
        driver = webdriver.Firefox()  # Start a new Firefox browser for each batch
        wait = WebDriverWait(driver, 3)  # Initialize WebDriverWait for the new driver instance
        batch = current_season_match_details[i:i + BATCH_SIZE]

        for j, match_info in enumerate(batch):
            total_index = i + j
            # Get the base URL (without hash fragment) to construct stats and commentary URLs
            base_match_url = match_info["Match URL"].split('#')[0]

            home_team = match_info["Home Team"]
            away_team = match_info["Away Team"]
            match_date_time = match_info["Match Date Time"]
            current_season_name = match_info["Season"]

            print(f"Processing match {total_index+1}/{len(current_season_match_details)} ({current_season_name}): {home_team} vs {away_team} ({match_date_time})")

            # --- Commentary Scraping ---
            url_for_commentary = base_match_url + "#/resumen-del-partido/comentarios-en-directo/0"

            current_match_commentary = {
                "Season": current_season_name,
                "Match URL": match_info["Match URL"],
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time,
                "Commentary": ""
            }

            try:
                driver.get(url_for_commentary)
                # Wait for the main commentary section to load
                wait.until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
                )
                # Wait for at least one commentary entry to be present
                wait.until(
                    EC.presence_of_all_elements_located((By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]"))
                )
                time.sleep(2)  # Give a moment for content to fully render after initial load

                # Find all commentary entries
                commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

                if commentary_entries:
                    full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                    if full_commentary_text.strip():
                        current_match_commentary["Commentary"] = full_commentary_text
                    else:
                        print(f"  Found commentary containers but no text for {url_for_commentary}.")
                        current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
                else:
                    try:
                        # Check for explicit "No commentary" message
                        no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'No hay comentarios')]")
                        commentary_message = no_commentary_message.text.strip()
                        print(f"  No commentary for {url_for_commentary}. Message: '{commentary_message}'")
                        current_match_commentary["Commentary"] = commentary_message
                    except NoSuchElementException:
                        print(f"  No commentary elements or explicit message found for {url_for_commentary}.")
                        current_match_commentary["Commentary"] = "No commentary available for this match."
            except NoSuchElementException as e:
                print(f"  Error finding main container for commentary at {url_for_commentary}: {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Error: Main container not found ({str(e)})."
            except TimeoutException:
                print(f"  Timeout while loading commentary at {url_for_commentary}. Skipping commentary.")
                current_match_commentary["Commentary"] = "Timeout: Could not load commentary."
            except Exception as e:
                print(f"  Unexpected error at {url_for_commentary}: {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

            all_commentary_data.append(current_match_commentary)
            time.sleep(2)  # Delay before moving to the next match

        driver.quit()  # Close the browser after each batch

# Step 4: Filter and save only matches with meaningful Spanish commentary
print("\n--- Saving Spanish commentary only ---")

df_commentary = pd.DataFrame(all_commentary_data)

# Drop rows with missing or trivial commentary
df_commentary = df_commentary[df_commentary['Commentary'].notna()]
df_commentary = df_commentary[df_commentary['Commentary'].str.strip() != '']
df_commentary = df_commentary[~df_commentary['Commentary'].str.contains("No hay comentarios|No commentary available|Timeout|Error", case=False)]

# Select only relevant columns
spanish_commentary_df = df_commentary[['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'Commentary']]

# Save to CSV
output_filename = "flashscore_premier_league_commentary_ES.csv"
spanish_commentary_df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\nDone: Saved Spanish commentary for all matches to {output_filename}")
print(f"\nFirst 5 rows of the Spanish commentary DataFrame:")
# This part requires an environment where display is available (e.g., Jupyter Notebook)
# For a standard Python script, you might just print the head
print(spanish_commentary_df.head().to_string())
print(f"\nShape of the Spanish commentary DataFrame: {spanish_commentary_df.shape}")


--- Starting to scrape season: 2017-2018 ---
No more 'show more' button found or timed out for 2017-2018. All matches loaded.
Found 380 match links with details for 2017-2018.

[INFO] Restarting browser... Batch starting from match 1 of 380
Processing match 1/380 (2017-2018): Burnley vs Bournemouth (13.05. 07:00)
Processing match 2/380 (2017-2018): Crystal Palace vs West Brom (13.05. 07:00)
Processing match 3/380 (2017-2018): Huddersfield vs Arsenal (13.05. 07:00)
Processing match 4/380 (2017-2018): Liverpool vs Brighton (13.05. 07:00)
Processing match 5/380 (2017-2018): Manchester Utd vs Watford (13.05. 07:00)
Processing match 6/380 (2017-2018): Newcastle vs Chelsea (13.05. 07:00)
Processing match 7/380 (2017-2018): Southampton vs Manchester City (13.05. 07:00)
Processing match 8/380 (2017-2018): Swansea vs Stoke (13.05. 07:00)
Processing match 9/380 (2017-2018): Tottenham vs Leicester (13.05. 07:00)
Processing match 10/380 (2017-2018): West Ham vs Everton (13.05. 07:00)
Processing m

KeyboardInterrupt: 